In [ ]:
# run this notebook after 12_analyze_trio_quad_standard_QC_R
# use this notebook to use quads and determine ibd2 trim and minimum ibd2 length
# after this, run 14_analyze_siblings_standard_QC_ibd2_R

In [ ]:
library(dplyr)
library(data.table)
library(ggplot2)

In [ ]:
all_snps_giab_filtered = fread("./quad_sibs_all_snps_after_standard_QC_GIAB.txt")
quad = fread("./relatedness/quads_new.txt", sep = "\t")
flagged = fread("./flagged_samples.tsv", sep = "\t", header= TRUE)
quad = quad %>% filter(!(off1 %in% flagged$s) & !(off2 %in% flagged$s) & !(par1 %in% flagged$s) & !(par2 %in% flagged$s) )

In [ ]:
trio_difs = fread("./trios_snps_after_standard_QC_GIAB.txt", sep = "\t")

In [ ]:
# cm = 0 
# subset sib difs and trio difs to those in ibd2 for given trim and merge 

ibd_file = fread("./ibd2_all.txt", sep = "\t")
quad$cm_0_num_sib_difs = NA

quad$cm_0_num_sib_difs_in_trio = NA
for (i in 1:nrow(quad)){
    i1 = quad$off1[i]
    i2 = quad$off2[i]
    idvs = c(i1, i2)
    # subset ibd file to those individuals 
    ibd_pair = ibd_file %>% filter(ID1 == i1 & ID2 == i2)
    # turn ibd file to bed file 
    ibd_bed = data.frame(chrom = ibd_pair$Chr, start = ibd_pair$start_coordinate, end = ibd_pair$stop_coordinate)
    ibd_bed = ibd_bed %>% filter(end - start > 1)
    # subset trio file to those individuals 
    trio_pair = trio_difs %>% filter(off %in% idvs)
    # turn trio file to bed file
    trio_bed = data.frame(chrom = trio_pair$chr, start = trio_pair$pos, end = trio_pair$pos+1, idv = trio_pair$off)
    # subset sib file to those individuals 
    sib_pair = all_snps_giab_filtered %>% filter(idv1 == i1 & idv2 == i2)
    # turn sib file to bed file 
    sib_bed = data.frame(chrom = sib_pair$chr, start = sib_pair$pos,
                        end = sib_pair$pos+1)
    # subset sib file to those in ibd bed 
    merged_bed = bedtoolsr::bt.intersect(a = sib_bed, b = ibd_bed, c = TRUE)
    sib_bed$idv = sib_pair$idv
    sib_ibd = sib_bed[which(merged_bed$V4 != 0),]
    quad$cm_0_num_sib_difs[i] = nrow(sib_ibd)
    # count the number of sib difs that are also trio difs 
    # there is only one difference per locus, so just merge on locus 
    trio_bed$in_trio = TRUE
    sib_in_trio = merge(sib_ibd, trio_bed, by = c("chrom", "start", "end"), all.x = TRUE)
    sib_in_trio = unique(sib_in_trio)
    quad$cm_0_num_sib_difs_in_trio[i] = sum(!is.na(sib_in_trio$in_trio))
}

In [ ]:
# cm = 0.25 
# subset sib difs and trio difs to those in ibd2 for given trim and merge 

ibd_file = fread("./ibd2_all_trimmed_0.25cM.txt", sep = "\t")
quad$cm_0.25_num_sib_difs = NA
quad$cm_0.25_num_sib_difs_in_trio = NA
for (i in 1:nrow(quad)){
    i1 = quad$off1[i]
    i2 = quad$off2[i]
    idvs = c(i1, i2)
    # subset ibd file to those individuals 
    ibd_pair = ibd_file %>% filter(ID1 == i1 & ID2 == i2)
    # turn ibd file to bed file 
    ibd_bed = data.frame(chrom = ibd_pair$Chr, start = ibd_pair$new_start_bp, end = ibd_pair$new_end_bp)
    ibd_bed = ibd_bed %>% filter(end - start > 1)
    # subset trio file to those individuals 
    trio_pair = trio_difs %>% filter(off %in% idvs)
    # turn trio file to bed file
    trio_bed = data.frame(chrom = trio_pair$chr, start = trio_pair$pos, end = trio_pair$pos+1)
    # subset sib file to those individuals 
    sib_pair = all_snps_giab_filtered %>% filter(idv1 == i1 & idv2 == i2)
    # turn sib file to bed file 
    sib_bed = data.frame(chrom = sib_pair$chr, start = sib_pair$pos,
                        end = sib_pair$pos+1)
    # subset sib file to those in ibd bed 
    merged_bed = bedtoolsr::bt.intersect(a = sib_bed, b = ibd_bed, c = TRUE)
    sib_bed$idv = sib_pair$idv
    sib_ibd = sib_bed[which(merged_bed$V4 != 0),]
    quad$cm_0.25_num_sib_difs[i] = nrow(sib_ibd)
    # there is only one difference per locus, so just merge on locus 
    trio_bed$in_trio = TRUE
    sib_in_trio = merge(sib_ibd, trio_bed, by = c("chrom", "start", "end"), all.x = TRUE)
    sib_in_trio = unique(sib_in_trio)
    quad$cm_0.25_num_sib_difs_in_trio[i] = sum(!is.na(sib_in_trio$in_trio))
}

In [ ]:
# cm = 0.5 
# subset sib difs and trio difs to those in ibd2 for given trim and merge 

ibd_file = fread("./ibd2_all_trimmed_0.5cM.txt", sep = "\t")
quad$cm_0.5_num_sib_difs = NA
quad$cm_0.5_num_sib_difs_in_trio = NA
for (i in 1:nrow(quad)){
    i1 = quad$off1[i]
    i2 = quad$off2[i]
    idvs = c(i1, i2)
    # subset ibd file to those individuals 
    ibd_pair = ibd_file %>% filter(ID1 == i1 & ID2 == i2)
    # turn ibd file to bed file 
    ibd_bed = data.frame(chrom = ibd_pair$Chr, start = ibd_pair$new_start_bp, end = ibd_pair$new_end_bp)
    ibd_bed = ibd_bed %>% filter(end - start > 1)
    # subset trio file to those individuals 
    trio_pair = trio_difs %>% filter(off %in% idvs)
    # turn trio file to bed file
    trio_bed = data.frame(chrom = trio_pair$chr, start = trio_pair$pos, end = trio_pair$pos+1)
    # subset sib file to those individuals 
    sib_pair = all_snps_giab_filtered %>% filter(idv1 == i1 & idv2 == i2)
    # turn sib file to bed file 
    sib_bed = data.frame(chrom = sib_pair$chr, start = sib_pair$pos,
                        end = sib_pair$pos+1)
    # subset sib file to those in ibd bed 
    merged_bed = bedtoolsr::bt.intersect(a = sib_bed, b = ibd_bed, c = TRUE)
    sib_bed$idv = sib_pair$idv
    sib_ibd = sib_bed[which(merged_bed$V4 != 0),]
    quad$cm_0.5_num_sib_difs[i] = nrow(sib_ibd)
    # there is only one difference per locus, so just merge on locus 
    trio_bed$in_trio = TRUE
    sib_in_trio = merge(sib_ibd, trio_bed, by = c("chrom", "start", "end"), all.x = TRUE)
    sib_in_trio = unique(sib_in_trio)
    quad$cm_0.5_num_sib_difs_in_trio[i] = sum(!is.na(sib_in_trio$in_trio))
}

In [ ]:
ibd_file = fread("./ibd2_all_trimmed_0.75cM.txt", sep = "\t")
quad$cm_0.75_num_sib_difs = NA
quad$cm_0.75_num_sib_difs_in_trio = NA
for (i in 1:nrow(quad)){
    i1 = quad$off1[i]
    i2 = quad$off2[i]
    idvs = c(i1, i2)
    # subset ibd file to those individuals 
    ibd_pair = ibd_file %>% filter(ID1 == i1 & ID2 == i2)
    # turn ibd file to bed file 
    ibd_bed = data.frame(chrom = ibd_pair$Chr, start = ibd_pair$new_start_bp, end = ibd_pair$new_end_bp)
    ibd_bed = ibd_bed %>% filter(end - start > 1)
    # subset trio file to those individuals 
    trio_pair = trio_difs %>% filter(off %in% idvs)
    # turn trio file to bed file
    trio_bed = data.frame(chrom = trio_pair$chr, start = trio_pair$pos, end = trio_pair$pos+1)
    # subset sib file to those individuals 
    sib_pair = all_snps_giab_filtered %>% filter(idv1 == i1 & idv2 == i2)
    # turn sib file to bed file 
    sib_bed = data.frame(chrom = sib_pair$chr, start = sib_pair$pos,
                        end = sib_pair$pos+1)
    # subset sib file to those in ibd bed 
    merged_bed = bedtoolsr::bt.intersect(a = sib_bed, b = ibd_bed, c = TRUE)
    sib_bed$idv = sib_pair$idv
    sib_ibd = sib_bed[which(merged_bed$V4 != 0),]
    quad$cm_0.75_num_sib_difs[i] = nrow(sib_ibd)
    # there is only one difference per locus, so just merge on locus 
    trio_bed$in_trio = TRUE
    sib_in_trio = merge(sib_ibd, trio_bed, by = c("chrom", "start", "end"), all.x = TRUE)
    sib_in_trio = unique(sib_in_trio)
    quad$cm_0.75_num_sib_difs_in_trio[i] = sum(!is.na(sib_in_trio$in_trio))
}

In [ ]:
ibd_file = fread("./ibd2_all_trimmed_1cM.txt", sep = "\t")
quad$cm_1_num_sib_difs = NA
quad$cm_1_num_sib_difs_in_trio = NA
for (i in 1:nrow(quad)){
    i1 = quad$off1[i]
    i2 = quad$off2[i]
    idvs = c(i1, i2)
    # subset ibd file to those individuals 
    ibd_pair = ibd_file %>% filter(ID1 == i1 & ID2 == i2)
    # turn ibd file to bed file 
    ibd_bed = data.frame(chrom = ibd_pair$Chr, start = ibd_pair$new_start_bp, end = ibd_pair$new_end_bp)
    ibd_bed = ibd_bed %>% filter(end - start > 1)
    # subset trio file to those individuals 
    trio_pair = trio_difs %>% filter(off %in% idvs)
    # turn trio file to bed file
    trio_bed = data.frame(chrom = trio_pair$chr, start = trio_pair$pos, end = trio_pair$pos+1)
    # subset sib file to those individuals 
    sib_pair = all_snps_giab_filtered %>% filter(idv1 == i1 & idv2 == i2)
    # turn sib file to bed file 
    sib_bed = data.frame(chrom = sib_pair$chr, start = sib_pair$pos,
                        end = sib_pair$pos+1)
    # subset sib file to those in ibd bed 
    merged_bed = bedtoolsr::bt.intersect(a = sib_bed, b = ibd_bed, c = TRUE)
    sib_bed$idv = sib_pair$idv
    sib_ibd = sib_bed[which(merged_bed$V4 != 0),]
    quad$cm_1_num_sib_difs[i] = nrow(sib_ibd)
    # there is only one difference per locus, so just merge on locus 
    trio_bed$in_trio = TRUE
    sib_in_trio = merge(sib_ibd, trio_bed, by = c("chrom", "start", "end"), all.x = TRUE)
    sib_in_trio = unique(sib_in_trio)
    quad$cm_1_num_sib_difs_in_trio[i] = sum(!is.na(sib_in_trio$in_trio))
}

In [ ]:
ibd2_trim_data = data.frame(
    trim = c(0, 0.25, 0.5, 0.75, 1),
                           proportion = c(
                           sum(quad$cm_0_num_sib_difs_in_trio)/sum(quad$cm_0_num_sib_difs),
                           sum(quad$cm_0.25_num_sib_difs_in_trio)/sum(quad$cm_0.25_num_sib_difs),
                           sum(quad$cm_0.5_num_sib_difs_in_trio)/sum(quad$cm_0.5_num_sib_difs),
                           sum(quad$cm_0.75_num_sib_difs_in_trio)/sum(quad$cm_0.75_num_sib_difs),
                           sum(quad$cm_1_num_sib_difs_in_trio)/sum(quad$cm_1_num_sib_difs)    
                           ))
ibd2_trim_data$source = "aou"
ibd2_trim_data_ukb = data.frame(trim = c(0, 0.25, 0.5, 0.75, 1),
                           proportion = c(0.1861, 0.4391, 0.4906, 0.5363, 0.5366)) # this was calculated separately
ibd2_trim_data_ukb$source="ukb"
ibd2_trim_data = rbind(ibd2_trim_data, ibd2_trim_data_ukb)

In [ ]:
ggplot(data = ibd2_trim_data, aes(x = trim, y = proportion, color = source)) + geom_point(size = 3) + 
theme_bw(base_size = 12) + labs(x = "Trim (cM)", y = "Proportion of sib differences in trio") 

In [ ]:
sib_in_trio = function(sib_file, trio_file, ibd_bed){
    if (nrow(ibd_bed) != 0){
        # turn sib file to bed file 
        sib_bed = data.frame(chrom = sib_file$chr, start = sib_file$pos,
                        end = sib_file$pos+1)
        # subset sib file to those in ibd bed 
        merged_bed = bedtoolsr::bt.intersect(a = sib_bed, b = ibd_bed, c = TRUE)
        sib_ibd = sib_bed[which(merged_bed$V4 != 0),]
        if (nrow(sib_ibd) != 0){
            # turn trio file to bed file
            trio_bed = data.frame(chrom = trio_file$chr, start = trio_file$pos, end = trio_file$pos+1)
            # subset trio file to those in ibd bed 
            merged_bed = bedtoolsr::bt.intersect(a = trio_bed, b = ibd_bed, c = TRUE)
            trio_ibd = trio_bed[which(merged_bed$V4 != 0),]
            if (nrow(trio_ibd) != 0){ # if there are any trio difs in these ibd2 regions
                trio_ibd$in_trio = TRUE
                s_in_t = merge(sib_ibd, trio_ibd, by = c("chrom", "start", "end"), all.x = TRUE)
                return(c(nrow(sib_ibd), sum(!is.na(s_in_t$in_trio))))
            } else { # if no trio difs in these ibd2 regions, return 0 sib_in_trio
                return(c(nrow(sib_ibd),0))
            }
        } else { # if no sib difs in these ibd regions, return 0,0
            return(c(0,0))
        }
    } else { # if there are no ibd segments in this category, return 0,0 
       return(c(0,0)) 
    }
}


In [ ]:
# determine minimum length for ibd2 segment 
ibd_file = fread("./ibd2_all_trimmed_0.75cM.txt", sep = "\t")
labels = c("<2cm", "2cm-4cm", "4cm-6cm", "6cm-10cm", "10cm-20cm", ">=20cm")
lengths = c(2, 4, 6,
           10, 20)
tot_sib = rep(0, length(labels))
tot_sib_in_trio = rep(0, length(labels))
for (i in 1:nrow(quad)){
    if (i %% 10 == 0){
        message(i)
    }
    i1 = quad$off1[i]
    i2 = quad$off2[i]
    idvs = c(i1, i2)
    # subset ibd file to those individuals 
    ibd_pair = ibd_file %>% filter(ID1 == i1 & ID2 == i2)
    ibd_pair = ibd_pair %>% filter(new_end_bp - new_start_bp > 1)
    # turn ibd file to bed file 
    ibd_bed = data.frame(chrom = ibd_pair$Chr, start = ibd_pair$new_start_bp, end = ibd_pair$new_end_bp)
    ibd_bed$length_cm = ibd_pair$length_cm
    # subset trio file to those individuals 
    trio_pair = trio_difs %>% filter(off %in% idvs)
    # subset sib file to those individuals 
    sib_pair = all_snps_giab_filtered %>% filter(idv1 == i1 & idv2 == i2)
    # go across each ibd segment length 
    ibd_bed_pair = ibd_bed %>% filter(length_cm < lengths[1])
    res = sib_in_trio(sib_pair, trio_pair, ibd_bed_pair)
    tot_sib[1] = tot_sib[1] + res[1]
    tot_sib_in_trio[1] = tot_sib_in_trio[1] + res[2]
    for (l in 1:(length(lengths)-1)){
        ibd_bed_pair = ibd_bed %>% filter(length_cm >= lengths[l] & length_cm < lengths[l+1])
        res = sib_in_trio(sib_pair, trio_pair, ibd_bed_pair)
        tot_sib[l+1] = tot_sib[l+1] + res[1]
        tot_sib_in_trio[l+1] = tot_sib_in_trio[l+1] + res[2]
    }
    # go across each ibd segment length 
    ibd_bed_pair = ibd_bed %>% filter(length_cm >= lengths[length(lengths)])
    res = sib_in_trio(sib_pair, trio_pair, ibd_bed_pair)
    tot_sib[length(labels)] = tot_sib[length(labels)] + res[1]
    tot_sib_in_trio[length(labels)] = tot_sib_in_trio[length(labels)] + res[2]
}

In [ ]:
minimum_length_plot = data.frame(labs = factor(labels, levels = labels),
                                proportion = tot_sib_in_trio/tot_sib)

In [ ]:
ggplot(data = minimum_length_plot, aes(x = labs, y = proportion)) + geom_point(fill = "black",size=3) + 
theme_bw(base_size = 12) + 
labs(x = "length", y = "Proportion of sibling differences identified in the trio analysis\nin IBD2 segments with length < x") 
